In [22]:
import cobra 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns   
from cobra.flux_analysis import loopless_solution
import json


# metabolic modeling of cyanobacteria - FBA
## Goal : compare metabolism of (i) WT cyanobacteria, (ii) a mutant with native phosphorespiration pathway deleted (dCCM), and (iii) the mutant dCCM with BAHC reactions added (pathway to recycle C2 compound produced by rubisco)
### We will use the previously published metabolic network iMS837 to simulate fluxes in the WT, and modify the network to model the (ii) and (iii) cases


In this notebook, we create the mutant dCCM : 
- remove the carboxysome 
- create a new Ribulose-bisphosphate carboxylase/oxygenase - cytosolic, with a 3:1 co2/o2 ratio 


In [2]:
BASE_FOLDER='/home/nathan/PhD/04.Computational/202609_BHAC_cyano/'
INPUT_FOLDER=f'{BASE_FOLDER}1_data/'
RESULTS_FOLDER=f'{BASE_FOLDER}3_results/'

Load cobra model 

In [3]:
model = cobra.io.load_json_model(f'{INPUT_FOLDER}iMS837.json')

Set parameter Username
Set parameter LicenseID to value 2749904
Academic license - for non-commercial use only - expires 2026-12-04


Set bounds on exchange reactions to simulate the proper experimental conditions

In [16]:
### nitrate is the only N-source -> no ammonium/ammonia
model.reactions.EX_nh4_e.lower_bound = 0
## we will constrain atmospheric CO2 in %, so we block hco3 uptake 
model.reactions.EX_hco3_e.lower_bound = 0

##modify co2 bound ? 
model.reactions.EX_co2_e.lower_bound = -10

create mutant : delete all reactions in carboxysome and replace 99:1 rubisco reaction with a 3:1 co2/o2 one 

In [17]:
# model.reactions.remove_reactions([r.id for r in model.reactions if 'cx' in r.compartments])
rxn_cx=list(set([r for r in model.reactions for m in r.metabolites if 'carboxyzome' in m.compartment ]))
##remove these from the model 
model.remove_reactions([r.id for r in rxn_cx],remove_orphans=True)

In [18]:
## add cytosolic rubisco reaction
RBPCc=cobra.Reaction('RBPCc')
model.add_reactions([RBPCc])
##add mets  4 rb15bp_c + 3 co2_c + 3 h2o_c + 1 o2_c --> 7 3pg_c + 1 2pglyc_c + 8 h_c
RBPCc.add_metabolites({'rb15bp_c':-4,'co2_c':-3,'h2o_c':-3,'o2_c':-1,'3pg_c':7,'2pglyc_c':1,'h_c':8})



Ignoring reaction 'RBPCc' since it already exists.


ValueError: Reaction 'RBPCc' does not belong to a model. Either add the reaction to a model or use Metabolite objects instead of strings as keys.

In [19]:
###biomass is model.reactions.BOF
model.objective='BOF'

In [8]:
###loopless FBA

solution = model.optimize()          # regular FBA first, to get the objective
sol = loopless_solution(model, fluxes=solution.fluxes)  # remove thermodynamically infeasible internal loops
dfsol=sol.fluxes.to_frame()

In [23]:
##save dsol as json for visualization with escher
with open(f'{RESULTS_FOLDER}iMS837_dCCM_dfsol.json', 'w') as f:
    json.dump(dfsol.to_dict(), f)

analyze flux of reactions that have at least one metabolite in the carboxysome 

In [10]:
### glyclt_c is glycolate , ### RBPCcx is rubisco reaction
## we are interested in the fluxes that consume produces the following metabolites 
## glycine gly_c, serine ser__L_c, aspartate asp__L_c
rxn_13cmets=list(set([r.id for r in model.reactions for m in r.metabolites if m.id in ['gly_c','ser__L_c','asp__L_c']]))
##decompose in 3 lists for each mthat have at least one metabolite in the carboxysome et 
rxn_ser= [r.id for r in model.reactions if 'ser__L_c' in [m.id for m in r.metabolites]]
rxn_gly= [r.id for r in model.reactions if 'gly_c' in [m.id for m in r.metabolites]]
rxn_asp= [r.id for r in model.reactions if 'asp__L_c' in [m.id for m in r.metabolites]]
rxn_glx= [r.id for r in model.reactions if 'glx_c' in [m.id for m in r.metabolites]]

In [11]:
dfsol.loc[rxn_glx]
dfsol.loc[['AGTi','SPTr','EX_co2_e']]

,fluxes
AGTi,3.506606
SPTr,-3.689887
EX_co2_e,-8.520529


turnover of metabolite e.g. nadh

In [12]:
def flux_of_metabolite(S_df, fluxes_df:pd.Series, MET:str,lb_abs=1e-1)->pd.DataFrame:
    """given the stoicheometry matrix, the flux dataframe for all reactions and the metabolite we look for 
        we get the values of fluxes that produce or consume this mets, 
        returns a dataframe with the fluxes values of the reactions that produce or consume the metabolite"""
    S_T_df = S_df.T
    ##MET_rxns_stoich is a series of reactions with how much of the metabolite is produced or consumed
    MET_rxns_stoich=S_T_df[MET]
    MET_rxns_stoich=MET_rxns_stoich[MET_rxns_stoich!=0] ##drop 0 values : rxn that dont produce or consume the metabolite
    #multiply fluxes values by the stoicheometry of the metabolite in the reaction
    df_met_flux=fluxes_df.loc[MET_rxns_stoich.index].mul(MET_rxns_stoich, axis=0)
    # drop low values
    df_met_flux=df_met_flux[abs(df_met_flux)>lb_abs] 
    return df_met_flux ##get the fluxes values of the reactions that produce or consume the metabolite
S=cobra.util.array.create_stoichiometric_matrix(model)
S_df=pd.DataFrame(S,columns=[r.id for r in model.reactions],index=[m.id for m in model.metabolites])


In [13]:
met='glu__L_c'
df_met=flux_of_metabolite(S_df,dfsol,met)
def filter_absthresh(df,abs_tresh=1):
    return df[abs(df)>abs_tresh]

dfplot=filter_absthresh(df_met).dropna()

In [14]:
display(dfplot)
model.reactions.PSERT

,fluxes
GLNS,-3.515372
GLUSfx,6.312106
PSERT,-2.139609


Reaction identifier,PSERT
Name,Phosphoserine transaminase
Memory address,0x7e43621190c0
Stoichiometry,3php_c + glu__L_c <=> akg_c + pser__L_c 3-Phosphonooxypyruvate + L-Glutamate <=> 2-Oxoglutarate + O-Phospho-L-serine
GPR,Synpcc7942_0191
Lower bound,-1000.0
Upper bound,1000.0


compare growth rate with different % of Co2

In [15]:
co2_levels = {'1pct': 1.0, '0.5pct': 0.5, '0.3pct': 0.3}
reference_pct = 0.04          # e.g. ambient air, your calibration point
reference_bound = 2.0         # mmol/gDW/h — the CO2 uptake bound you've validated growth against at reference_pct

for label, pct in co2_levels.items():
    scaled_bound = reference_bound * (pct / reference_pct)
    with model:  # context manager: change reverts after the block
        model.exchanges.get_by_id('EX_co2_e').lower_bound = -scaled_bound
        sol = model.optimize()
        print(label, '-> CO2 bound:', -scaled_bound, '| growth:', sol.objective_value)

1pct -> CO2 bound: -50.0 | growth: 0.2187355095909488
0.5pct -> CO2 bound: -25.0 | growth: 0.2187355095909485
0.3pct -> CO2 bound: -15.0 | growth: 0.2187355095909485
